# EXPERTO 2 — Osteoarthritis Knee X-ray
Clasificación de grados KL: 0=Normal, 1=Dudoso, 2=Leve,
                              3=Moderado, 4=Severo
Arquitectura: EfficientNet-B0 (liviano, ideal para 4K imágenes)
Loss: CrossEntropyLoss
Métrica objetivo: F1 Macro > 0.72
Hardware: RTX 3050 6GB local


In [1]:
import os
from pathlib import Path
import json
 
import os, random, warnings, time
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from pathlib import Path
from PIL import Image
from sklearn.metrics import f1_score, confusion_matrix
 
warnings.filterwarnings("ignore")

In [2]:
os.environ["KAGGLE_CONFIG_DIR"] = r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\.kaggle"

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()
print("Autenticado OK")

# Ver tamaño del dataset antes de descargar
api.dataset_list_files("dhruvacube/osteoarthritis")

Autenticado OK


{"datasetFiles": [{"ref": "", "datasetRef": "", "ownerRef": "", "name": "KLGrade/KLGrade/0/0_110_png.rf.640ac861f605968e142166b62b883aa2.jpg", "creationDate": "2024-11-13T12:26:05.201Z", "description": "", "fileType": "", "url": "", "totalBytes": 29300, "columns": []}, {"ref": "", "datasetRef": "", "ownerRef": "", "name": "KLGrade/KLGrade/0/0_111_png.rf.feb50161b2231dc9244620846a45de3e.jpg", "creationDate": "2024-11-13T12:26:05.197Z", "description": "", "fileType": "", "url": "", "totalBytes": 30029, "columns": []}, {"ref": "", "datasetRef": "", "ownerRef": "", "name": "KLGrade/KLGrade/0/0_112_png.rf.f0ab25148858fd1db3bdd5934037c7ae.jpg", "creationDate": "2024-11-13T12:26:05.204Z", "description": "", "fileType": "", "url": "", "totalBytes": 23949, "columns": []}, {"ref": "", "datasetRef": "", "ownerRef": "", "name": "KLGrade/KLGrade/0/0_114_png.rf.c06380e33be324db4c96b3e0a072b47e.jpg", "creationDate": "2024-11-13T12:26:05.202Z", "description": "", "fileType": "", "url": "", "totalBytes

In [6]:
# Descargar dataset
DOWNLOAD_DIR = r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\datasets\osteo"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("Descargando Osteoarthritis (~150 MB)...")
api.dataset_download_files("dhruvacube/osteoarthritis", 
                           path=DOWNLOAD_DIR, 
                           unzip=True)
print("Descarga completa")

# Verificar estructura
for root, dirs, files in os.walk(DOWNLOAD_DIR):
    depth = root.replace(DOWNLOAD_DIR, '').count(os.sep)
    if depth > 2: continue
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/")
    if depth == 2:
        print(f"{indent}  ({len(files)} archivos)")

Descargando Osteoarthritis (~150 MB)...
Dataset URL: https://www.kaggle.com/datasets/dhruvacube/osteoarthritis


KeyboardInterrupt: 

In [3]:
DOWNLOAD_DIR = r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\datasets\osteo"

osteo_root = Path(DOWNLOAD_DIR) / "KLGrade" / "KLGrade"
for class_dir in sorted(osteo_root.iterdir()):
    if class_dir.is_dir():
        n = len(list(class_dir.iterdir()))
        print(f"  Clase {class_dir.name}: {n} imágenes")

  Clase 0: 1315 imágenes
  Clase 1: 1266 imágenes
  Clase 2: 765 imágenes
  Clase 3: 742 imágenes
  Clase 4: 678 imágenes


In [4]:
# ── Seeds ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
 

Dispositivo : cuda
GPU         : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM        : 6.4 GB


In [5]:
# ── Rutas ────────────────────────────────────────────────────────
OSTEO_ROOT = Path(r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\datasets\osteo\KLGrade\KLGrade")
OUT_DIR    = Path(r"C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\checkpoints")
OUT_DIR.mkdir(parents=True, exist_ok=True)
 
LABELS    = ["Normal", "Dudoso", "Leve", "Moderado", "Severo"]
N_CLASSES = 5
 
print(f"\nOsteo root existe: {OSTEO_ROOT.exists()}")
for class_dir in sorted(OSTEO_ROOT.iterdir()):
    if class_dir.is_dir():
        n = len(list(class_dir.iterdir()))
        print(f"  Clase {class_dir.name} ({LABELS[int(class_dir.name)]}): {n} imágenes")



Osteo root existe: True
  Clase 0 (Normal): 1315 imágenes
  Clase 1 (Dudoso): 1266 imágenes
  Clase 2 (Leve): 765 imágenes
  Clase 3 (Moderado): 742 imágenes
  Clase 4 (Severo): 678 imágenes


In [6]:
# ── Dataset ──────────────────────────────────────────────────────
class OsteoDataset(Dataset):
    """
    Osteoarthritis — KLGrade/KLGrade/{0,1,2,3,4}/*.jpg|png
    Clasificación de grados Kellgren-Lawrence (0-4).
    """
    def __init__(self, root, transform, split="train",
                 val_fraction=0.15):
        self.transform = transform
        valid_exts     = {".png", ".jpg", ".jpeg"}
 
        all_samples = []
        for class_dir in sorted(Path(root).iterdir()):
            if not class_dir.is_dir():
                continue
            try:
                kl = int(class_dir.name)
            except ValueError:
                continue
            for fpath in class_dir.iterdir():
                if fpath.suffix.lower() in valid_exts:
                    all_samples.append((fpath, kl))
 
        # Split estratificado por clase
        rng = random.Random(SEED)
        rng.shuffle(all_samples)
        n_val = int(len(all_samples) * val_fraction)
 
        if split == "train":
            samples = all_samples[n_val:]
        else:
            samples = all_samples[:n_val]
 
        self.paths  = [s[0] for s in samples]
        self.labels = [s[1] for s in samples]
 
        # Distribución
        from collections import Counter
        dist = Counter(self.labels)
        print(f"  Osteo ({split}): {len(self.paths)} imágenes | "
              f"{dict(sorted(dist.items()))}")
 
    def __len__(self): return len(self.paths)
 
    def __getitem__(self, idx):
        img   = Image.open(self.paths[idx]).convert("RGB")
        img   = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label
 

In [7]:
# ── Transforms ───────────────────────────────────────────────────
# CLAHE no está en torchvision — simulamos con RandomAutocontrast
# que mejora el contraste óseo de forma similar
TRANSFORM_TRAIN = T.Compose([
    T.Resize((256, 256), antialias=True),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAutocontrast(p=0.3),   # mejora contraste óseo
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])
 
TRANSFORM_VAL = T.Compose([
    T.Resize((224, 224), antialias=True),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])
 
print("\nCargando datasets...")
ds_train = OsteoDataset(OSTEO_ROOT, TRANSFORM_TRAIN, split="train")
ds_val   = OsteoDataset(OSTEO_ROOT, TRANSFORM_VAL,   split="val")
 
# Class weights para CrossEntropyLoss
# (clases 2,3,4 tienen menos muestras)
counts      = np.bincount([s for s in ds_train.labels], minlength=N_CLASSES)
class_w     = 1.0 / (counts + 1e-6)
class_w     = class_w / class_w.sum() * N_CLASSES   # normalizar
class_w_t   = torch.tensor(class_w, dtype=torch.float32).to(DEVICE)
print(f"\nClass weights: {class_w.round(3)}")
 
# DataLoaders — batch 32 para RTX 3050 6GB
loader_train = DataLoader(ds_train, batch_size=32, shuffle=True,
                          num_workers=0,   # ← 0 en Windows+Jupyter
                          pin_memory=True,
                          persistent_workers=False)
loader_val   = DataLoader(ds_val,   batch_size=32, shuffle=False,
                          num_workers=0,   # ← 0 en Windows+Jupyter
                          pin_memory=True,
                          persistent_workers=False)
 
print(f"\nBatches train: {len(loader_train)}")
print(f"Batches val  : {len(loader_val)}")
 


Cargando datasets...
  Osteo (train): 4052 imágenes | {0: 1130, 1: 1074, 2: 639, 3: 628, 4: 581}
  Osteo (val): 714 imágenes | {0: 185, 1: 192, 2: 126, 3: 114, 4: 97}

Class weights: [0.661 0.695 1.169 1.189 1.286]

Batches train: 127
Batches val  : 23


In [8]:
# ── Modelo: EfficientNet-B0 ──────────────────────────────────────
# Más liviano que VGG-16 BN, mejor F1 en datasets pequeños.
# Con 4K imágenes, un modelo grande sobreajusta fácil.
print("\nCargando EfficientNet-B0 preentrenado...")
model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
)
 
# Reemplazar cabeza
in_features      = model.classifier[1].in_features   # 1280
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, N_CLASSES),
)
 
model = model.to(DEVICE)
 
total  = sum(p.numel() for p in model.parameters()) / 1e6
print(f"  Parámetros: {total:.1f}M")
print(f"  VRAM estimada: ~{total*4:.0f} MB (FP32) → ~{total*2:.0f} MB con FP16")
 


Cargando EfficientNet-B0 preentrenado...
  Parámetros: 4.0M
  VRAM estimada: ~16 MB (FP32) → ~8 MB con FP16


In [9]:
# ── Entrenamiento ────────────────────────────────────────────────
EPOCHS      = 30        # más épocas porque el dataset es pequeño
ACCUM_STEPS = 2         # batch efectivo = 32 * 2 = 64
LR_FEATURES = 5e-5
LR_HEAD     = 5e-4
 
criterion = nn.CrossEntropyLoss(weight=class_w_t)
scaler    = GradScaler(device="cuda" if DEVICE.type == "cuda" else "cpu")
 
optimizer = torch.optim.Adam([
    {"params": model.features.parameters(),    "lr": LR_FEATURES},
    {"params": model.classifier.parameters(),  "lr": LR_HEAD},
], weight_decay=1e-4)
 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)
 
 
def train_epoch(model, loader, optimizer, criterion, scaler, accum_steps):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    n = len(loader)

    for i, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast(device_type=DEVICE.type):
            logits = model(imgs)
            loss   = criterion(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * accum_steps

        # Progreso cada 20 batches
        if (i + 1) % 20 == 0 or (i + 1) == n:
            print(f"    batch {i+1}/{n}  loss: {total_loss/(i+1):.4f}",
                  flush=True)

    return total_loss / n
 
 
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
 
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        with autocast(device_type=DEVICE.type):
            logits = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.numpy())
 
    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    f1     = f1_score(labels, preds, average="macro", zero_division=0)
    f1_per = f1_score(labels, preds, average=None,    zero_division=0)
    return f1, f1_per, preds, labels
 

In [10]:
# Forzar modo alto rendimiento en GPU
torch.cuda.set_per_process_memory_fraction(0.9)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# Verificar que CUDA está activo correctamente
print(f"TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")

TF32 matmul: True
cuDNN benchmark: True


In [11]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

# Test de velocidad real
x = torch.randn(64, 3, 224, 224).cuda()
model_test = model  # ya está en cuda

import time
# Warm up
for _ in range(3):
    with torch.no_grad():
        _ = model_test(x)

# Medir
t = time.time()
for _ in range(10):
    with torch.no_grad():
        _ = model_test(x)
torch.cuda.synchronize()
print(f"10 forward passes: {time.time()-t:.3f}s")
print(f"Por batch: {(time.time()-t)/10*1000:.1f}ms")

NVIDIA GeForce RTX 3050 6GB Laptop GPU
12.1
10 forward passes: 1.595s
Por batch: 159.5ms


In [12]:


# ── Loop ─────────────────────────────────────────────────────────
print(f"\nEntrenando {EPOCHS} épocas  |  "
      f"batch efectivo={32*ACCUM_STEPS}  |  "
      f"FP16={'Sí' if DEVICE.type=='cuda' else 'No'}")
 
best_f1, best_epoch, history = 0.0, 0, []
 
for epoch in range(1, EPOCHS + 1):
    t0         = time.time()
    train_loss = train_epoch(model, loader_train, optimizer,
                             criterion, scaler, ACCUM_STEPS)
    f1, f1_per, _, _ = evaluate(model, loader_val)
    elapsed    = time.time() - t0
 
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    history.append({"epoch": epoch, "loss": train_loss, "f1": f1})
 
    print(f"  Época {epoch:2d}/{EPOCHS}  "
          f"loss: {train_loss:.4f}  "
          f"F1: {f1:.4f}  "
          f"LR: {current_lr:.1e}  "
          f"({elapsed:.0f}s)")
 
    if f1 > best_f1:
        best_f1, best_epoch = f1, epoch
        torch.save({
            "epoch"       : epoch,
            "model_state" : model.state_dict(),
            "f1_macro"    : f1,
            "f1_per_class": f1_per.tolist(),
            "labels"      : LABELS,
            "thresholds"  : None,   # single-label, no threshold needed
        }, OUT_DIR / "expert2_osteo_best.pth")
        print(f"    ✓ Guardado (F1={best_f1:.4f})")
 
print(f"\nMejor F1 Macro: {best_f1:.4f}  (época {best_epoch})")
if best_f1 >= 0.72:
    print("  ✓ Supera el umbral requerido (>0.72)")
elif best_f1 >= 0.65:
    print("  ~ Rango aceptable (>0.65)")
else:
    print("  ⚠ Por debajo del umbral")
 
 


Entrenando 30 épocas  |  batch efectivo=64  |  FP16=Sí
    batch 20/127  loss: 1.6014
    batch 40/127  loss: 1.5792
    batch 60/127  loss: 1.5551
    batch 80/127  loss: 1.5292
    batch 100/127  loss: 1.5036
    batch 120/127  loss: 1.4754
    batch 127/127  loss: 1.4631
  Época  1/30  loss: 1.4631  F1: 0.4239  LR: 5.0e-05  (52s)
    ✓ Guardado (F1=0.4239)
    batch 20/127  loss: 1.2922
    batch 40/127  loss: 1.2601
    batch 60/127  loss: 1.2250
    batch 80/127  loss: 1.1926
    batch 100/127  loss: 1.1699
    batch 120/127  loss: 1.1594
    batch 127/127  loss: 1.1535
  Época  2/30  loss: 1.1535  F1: 0.5703  LR: 4.9e-05  (36s)
    ✓ Guardado (F1=0.5703)
    batch 20/127  loss: 0.9486
    batch 40/127  loss: 0.9632
    batch 60/127  loss: 0.9736
    batch 80/127  loss: 0.9616
    batch 100/127  loss: 0.9545
    batch 120/127  loss: 0.9419
    batch 127/127  loss: 0.9381
  Época  3/30  loss: 0.9381  F1: 0.6386  LR: 4.9e-05  (33s)
    ✓ Guardado (F1=0.6386)
    batch 20/127  loss:

In [13]:
# ── Evaluación final detallada ───────────────────────────────────
print("\nCargando mejor checkpoint...")
ckpt = torch.load(OUT_DIR / "expert2_osteo_best.pth",
                  map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
 
f1, f1_per, preds, labels_arr = evaluate(model, loader_val)
 
print(f"\nF1 por clase (val):")
for label, f1c in zip(LABELS, f1_per):
    bar = "█" * int(f1c * 20)
    print(f"  {label:<10}: {f1c:.3f}  {bar}")
 
print(f"\nMatriz de confusión:")
cm = confusion_matrix(labels_arr, preds)
print(f"  {'':10}", end="")
for l in LABELS: print(f"{l[:6]:>8}", end="")
print()
for i, row in enumerate(cm):
    print(f"  {LABELS[i]:<10}", end="")
    for val in row: print(f"{val:>8}", end="")
    print()
 
print(f"\nF1 Macro final: {f1:.4f}")


Cargando mejor checkpoint...

F1 por clase (val):
  Normal    : 0.804  ████████████████
  Dudoso    : 0.735  ██████████████
  Leve      : 0.712  ██████████████
  Moderado  : 0.802  ████████████████
  Severo    : 0.940  ██████████████████

Matriz de confusión:
              Normal  Dudoso    Leve  Modera  Severo
  Normal         160      18       7       0       0
  Dudoso          43     132      15       2       0
  Leve            10      15      94       5       2
  Moderado         0       2      22      83       7
  Severo           0       0       0       3      94

F1 Macro final: 0.7987


In [14]:
# Guardar historial
with open(OUT_DIR / "expert2_history.json", "w") as f:
    json.dump({
        "history"      : history,
        "best_f1"      : best_f1,
        "best_epoch"   : best_epoch,
        "f1_per_class" : {l: float(v) for l, v in zip(LABELS, f1_per)},
        "architecture" : "EfficientNet-B0",
        "dataset"      : "Osteoarthritis KLGrade",
        "expert_id"    : 2,
    }, f, indent=2)
 
print(f"\n✓ Experto 2 listo")
print(f"  Checkpoint: {OUT_DIR / 'expert2_osteo_best.pth'}")


✓ Experto 2 listo
  Checkpoint: C:\Users\isape\OneDrive\Escritorio\ANALITICA\Proyecto2\checkpoints\expert2_osteo_best.pth
